# Unit 1 solutions: measure the starter

These reference solutions race the starter with fixed seeds, derive each answer independently where they can (a prediction first, then the measurement), and check it with executable asserts.
Every scratch file goes into a fresh temporary directory.

In [ ]:
from pathlib import Path
from tempfile import mkdtemp

import matplotlib.pyplot as plt
import numpy as np
from racekit import write_variant
from racekit.cars import KinematicCar
from racekit.check_submission import check_submission
from racekit.measure import noise_aware_better, repeat_races
from racekit.plots import plot_speed_map, plot_telemetry, plot_track
from racekit.runner import CONTROL_DT, run_race
from racekit.telemetry import race_meta, read_csv, sector_splits, write_csv
from racekit.tracks import load_track

STARTER = Path("../../projects/racekit/agents/unit-01.py")
workdir = Path(mkdtemp(prefix="roar-unit-01-sol-"))
with open(STARTER) as handle:
    starter_source = handle.read()
print("scratch files go in", workdir)

## Exercise 1 - Read the submission ABI

`step()` reads three sensors itself: the location, the roll-pitch-yaw (for the yaw) and the velocity (for the speed).
`advance_index` only receives the location `step()` already read, so it adds no sensor.
The camera, occupancy-map and collision sensors are stored by the constructor but never read.
The file annotates the waypoint list with `typing.List`, which works on Python 3.8; `list[...]` would not.

In [ ]:
step_sensors = ["location_sensor", "rpy_sensor", "velocity_sensor"]
action_keys = {"throttle", "steer", "brake", "hand_brake", "reverse", "target_gear"}
lookahead = 3
target_speed = 20.0
coroutines = ["initialize", "step"]
py38_safe_annotation = "List[roar_py_interface.RoarPyWaypoint]"

# Check every answer against the file's text.
all_sensors = [
    "camera_sensor", "location_sensor", "velocity_sensor", "rpy_sensor",
    "occupancy_map_sensor", "collision_sensor",
]
read_sensors = sorted(name for name in all_sensors if f"self.{name}.get_last_gym_observation()" in starter_source)
assert step_sensors == sorted(step_sensors) and step_sensors == read_sensors
assert all(f'"{key}":' in starter_source for key in action_keys) and len(action_keys) == 6
assert f"LOOKAHEAD_WAYPOINTS = {lookahead}" in starter_source
assert f"TARGET_SPEED = {target_speed}" in starter_source
methods = ["initialize", "nearest_index", "advance_index", "step"]
assert coroutines == sorted(name for name in methods if f"async def {name}(" in starter_source)
assert f"maneuverable_waypoints: {py38_safe_annotation}," in starter_source
assert "list[" not in py38_safe_annotation
step_sensors, action_keys, lookahead, target_speed, coroutines, py38_safe_annotation

## Exercise 2 - Race the starter on every track

The prediction is wrong in the same direction on every track, for two reasons.
A proportional speed controller never reaches its target against drag: the car settles near 19.5 m/s, not 20 m/s (Exercise 6 computes it).
And the race starts from rest, so the first few seconds are spent accelerating.
The error grows with the track length because the 0.5 m/s shortfall is paid on every metre.

In [ ]:
tracks = ["oval", "test-circuit", "monza-like"]
clean_times = {}
all_clean = True
predicted_times = {}
prediction_error = {}
for name in tracks:
    clean = run_race(STARTER, name, car=KinematicCar(jitter=0.0))
    clean_times[name] = clean.total_time
    all_clean = all_clean and clean.finished and clean.collisions == 0
    predicted_times[name] = 3 * load_track(name).length / 20.0
    prediction_error[name] = clean_times[name] - predicted_times[name]
why_slower = (
    "The proportional speed controller settles below 20 m/s (about 19.5 m/s), because a throttle "
    "of 0.05 x (20 - v) is zero at exactly 20 m/s and drag needs some throttle to hold any speed, "
    "and the race starts from rest, so the launch costs a few seconds on top."
)

for name in tracks:
    print(f"{name:13s} measured {clean_times[name]:7.2f} s, predicted {predicted_times[name]:7.2f} s, "
          f"error {prediction_error[name]:6.2f} s")
assert all_clean
assert abs(clean_times["oval"] - 223.50) < 1e-9  # the lesson's deterministic oval race
assert all(prediction_error[name] > 0 for name in tracks)
assert prediction_error["oval"] < prediction_error["test-circuit"] < prediction_error["monza-like"]
# Every total is a whole number of control ticks.
assert all(abs(clean_times[name] / CONTROL_DT - round(clean_times[name] / CONTROL_DT)) < 1e-6 for name in tracks)

## Exercise 3 - Query the telemetry

The throttle is `0.05 · (20 − v)`: 1.0 at rest, only 0.25 at 15 m/s and 0.05 at 19 m/s.
The push shrinks as the car closes in, while drag grows, so the speed creeps toward its settling point (about 19.5 m/s) like an exponential approach.
19 m/s is within 0.5 m/s of that point, so the last 4 m/s take longer than the first 15.

In [ ]:
race = run_race(STARTER, "monza-like", car=KinematicCar(jitter=0.05), seed=0)
telemetry = race.telemetry
t_15 = float(telemetry["t"][np.argmax(telemetry["v"] >= 15.0)])
t_19 = float(telemetry["t"][np.argmax(telemetry["v"] >= 19.0)])
v_max = float(np.max(telemetry["v"]))
fraction_above_19 = float(np.mean(telemetry["v"] >= 19.0))
max_abs_offset = float(np.max(np.abs(telemetry["offset"])))
launch_shape = (
    "The throttle is proportional to the remaining speed error, so it is large at rest but tiny near "
    "the target while drag keeps growing; the car approaches its settling speed of about 19.5 m/s "
    "more and more slowly, and 19 m/s is already close to it."
)

print("t_15 =", round(t_15, 2), "s | t_19 =", round(t_19, 2), "s | v_max =", round(v_max, 3), "m/s")
print("fraction >= 19 m/s:", round(fraction_above_19, 4), "| max |offset|:", round(max_abs_offset, 3), "m")
assert abs(race.total_time - 870.30) < 1e-9  # the lesson's race
assert np.max(telemetry["v"][telemetry["t"] < t_15]) < 15.0 <= telemetry["v"][np.argmax(telemetry["v"] >= 15.0)]
assert 0.0 < t_15 < t_19 and t_19 - t_15 > t_15  # 15 -> 19 is slower than 0 -> 15
assert 19.0 <= v_max < 20.5  # jitter can push a little past the target, never far
assert 0.95 < fraction_above_19 < 1.0
assert abs(max_abs_offset - 0.61) < 0.005  # the lesson's number

## Exercise 4 - Sector splits from a CSV

The CSV keeps six decimals, so the splits computed from it match the runner's to far better than a tick.
The nine splits tile the race from the start to the finish line, so they sum to the total time.

In [ ]:
oval_race = run_race(STARTER, "oval", car=KinematicCar(jitter=0.0))
csv_path = workdir / "oval.csv"
write_csv(oval_race.telemetry, csv_path, race_meta(oval_race))
loaded, meta = read_csv(csv_path)
csv_splits = sector_splits(loaded, meta["progress_total"], laps=meta["laps"], per_lap=meta["per_lap"])
max_split_difference = float(np.max(np.abs(csv_splits - oval_race.sector_splits)))
lap_times = [float(sum(lap)) for lap in csv_splits]
standing_start_penalty = float(csv_splits[0][0] - np.mean([csv_splits[1][0], csv_splits[2][0]]))
splits_total_matches = abs(float(np.sum(csv_splits)) - oval_race.total_time) < 1e-6

print(meta)
print(np.round(csv_splits, 2))
print("laps:", np.round(lap_times, 2), "| standing-start penalty:", round(standing_start_penalty, 2), "s")
assert csv_splits.shape == (3, 3) and meta["laps"] == 3 and meta["per_lap"] == 3
assert max_split_difference < 1e-6
assert len(lap_times) == 3 and abs(sum(lap_times) - oval_race.total_time) < 1e-6
assert lap_times[0] > lap_times[1] and lap_times[0] > lap_times[2]  # lap 1 carries the launch
assert 2.0 < standing_start_penalty < 5.0
assert splits_total_matches

## Exercise 5 - Repeated runs and the noise-aware rule

Both sets are the same file on the same car, so neither can genuinely beat the other, and the rule agrees.
Picking the slowest run of one set and the fastest of the other "shows" a gain larger than the whole spread.

By hand: `median(x) = 301.30`, `median(y) = 301.60`, a gap of 0.30 s.
The spreads are 301.45 − 301.20 = 0.25 s and 301.95 − 301.50 = 0.45 s.
Every `x` time is faster than every `y` time (the slowest `x`, 301.45, beats the fastest `y`, 301.50).
The rule still refuses: 0.30 s is not larger than 0.45 s, so `x` does **not** beat `y`.
That is deliberately conservative: with only five noisy runs, the rule asks for a median gap that clears the larger range, and accepts that it will sometimes say "not proven" about a real difference.

In [ ]:
jittered = lambda: KinematicCar(jitter=0.05)
runs_a = repeat_races(STARTER, "oval", n=5, car_factory=jittered)
runs_b = repeat_races(STARTER, "oval", n=5, seeds=[10, 11, 12, 13, 14], car_factory=jittered)
median_a, spread_a = runs_a.median, runs_a.spread
median_b, spread_b = runs_b.median, runs_b.spread
b_beats_a = noise_aware_better(runs_b.times, runs_a.times)
a_beats_b = noise_aware_better(runs_a.times, runs_b.times)
best_single_gap = float(max(runs_a.times) - min(runs_b.times))

x = [301.20, 301.45, 301.30, 301.25, 301.40]
y = [301.55, 301.95, 301.60, 301.70, 301.50]
gap = float(np.median(y) - np.median(x))
hand_x_beats_y = gap > max(max(x) - min(x), max(y) - min(y))
library_x_beats_y = noise_aware_better(x, y)

print("A:", runs_a, np.round(runs_a.times, 2))
print("B:", runs_b, np.round(runs_b.times, 2))
print("b beats a?", b_beats_a, "| a beats b?", a_beats_b, "| best single-run gap:", round(best_single_gap, 2), "s")
print("x vs y: median gap", round(gap, 2), "s; x beats y by hand?", hand_x_beats_y, "| library:", library_x_beats_y)
assert runs_a.all_finished and runs_b.all_finished and runs_a.collisions == 0 and runs_b.collisions == 0
assert abs(median_a - float(np.median(runs_a.times))) < 1e-9 and abs(spread_b - (max(runs_b.times) - min(runs_b.times))) < 1e-9
assert abs(median_a - 220.75) < 1e-9 and abs(spread_a - 0.35) < 1e-9  # the lesson's seeds 0-4
assert not b_beats_a and not a_beats_b
assert best_single_gap > max(spread_a, spread_b)  # one run of each would have lied
assert abs(gap - 0.30) < 1e-9
assert hand_x_beats_y == library_x_beats_y and not hand_x_beats_y

## Exercise 6 - Plot the speed controller's steady state

At a steady speed the throttle's push must cancel the drag: `6.0 · 0.05 · (20 − v) = 4e-4 · v²`, that is `4e-4 v² + 0.3 v − 6 = 0`.
The positive root is about 19.49 m/s.
A proportional controller only pushes when there is an error, and holding any speed against drag needs a push, so it must settle with a permanent error below the target.

In [ ]:
steady_race = run_race(STARTER, "oval", car=KinematicCar(jitter=0.0))
steady_tel = steady_race.telemetry
steady_speed = float(np.median(steady_tel["v"][steady_tel["t"] > 60.0]))
steady_state_error = 20.0 - steady_speed

fig, (ax_trace, ax_map) = plt.subplots(1, 2, figsize=(14, 5))
plot_telemetry(steady_tel, columns=("v",), ax=ax_trace)
ax_trace.axhline(20.0, color="tab:red", linestyle="--", linewidth=1.0, label="TARGET_SPEED (20 m/s)")
ax_trace.axhline(steady_speed, color="tab:green", linewidth=1.0, label=f"settles at {steady_speed:.2f} m/s")
ax_trace.set_xlabel("t (s)")
ax_trace.set_ylabel("v (m/s)")
ax_trace.set_title("Starter on the oval: speed settles below its target")
ax_trace.legend()
plot_speed_map(steady_tel, load_track("oval"), ax=ax_map)
ax_map.set_title("Speed over the oval (m/s)")
fig.tight_layout()
plt.show()

why_not_20 = (
    "A proportional controller's throttle 0.05 x (20 - v) is zero at 20 m/s, but drag 4e-4 v^2 always "
    "needs a positive throttle to hold speed, so the car settles where 6 x 0.05 x (20 - v) = 4e-4 v^2, "
    "about 19.49 m/s: the error is what produces the push."
)

# Solve the quadratic 4e-4 v^2 + 0.3 v - 6 = 0 independently.
qa, qb, qc = 4e-4, 6.0 * 0.05, -6.0 * 0.05 * 20.0
v_balance = float((-qb + np.sqrt(qb * qb - 4 * qa * qc)) / (2 * qa))
print("steady speed:", round(steady_speed, 4), "m/s | balance root:", round(v_balance, 4), "m/s | error:", round(steady_state_error, 4), "m/s")
assert steady_race.finished and steady_race.collisions == 0
assert abs(steady_speed - v_balance) < 0.01
assert 0.4 < steady_state_error < 0.6

## Exercise 7 - Spot the Python 3.9+ habit

Prediction, from the lesson's list of habits: `a` (`list[float]` annotation), `b` (`str.removeprefix`), `e` (`dict | dict`) and `f` (`match`).
`c` and `g` use `List` and `Optional`, which the starter already imports from `typing`, and `d` is the 3.5+ `{**a, **b}` merge.
`a` fails even though the starter has `from __future__ import annotations`: the checker refuses it anyway.
Each fix below is checked by appending it to the starter and running `check_submission` on 1 lap, to keep the notebook fast.

In [ ]:
CANDIDATES = {
    "a": "def mean_speed(speeds: list[float]) -> float:\n    return sum(speeds) / len(speeds)\n",
    "b": "def short_name(name: str) -> str:\n    return name.removeprefix('agent-')\n",
    "c": "def mean_speed(speeds: List[float]) -> float:\n    return sum(speeds) / len(speeds)\n",
    "d": "def merged(a: dict, b: dict) -> dict:\n    return {**a, **b}\n",
    "e": "def merged(a: dict, b: dict) -> dict:\n    return a | b\n",
    "f": (
        "def describe(speed: float) -> str:\n"
        "    match speed:\n"
        "        case 0.0:\n"
        "            return 'stopped'\n"
        "        case _:\n"
        "            return 'moving'\n"
    ),
    "g": "def limit(speed: float, cap: Optional[float] = None) -> float:\n    return speed if cap is None else min(speed, cap)\n",
}

candidate_paths = {}
for key, helper in CANDIDATES.items():
    candidate_paths[key] = workdir / f"candidate_{key}.py"
    with open(candidate_paths[key], "w") as handle:
        handle.write(starter_source + "\n\n" + helper)

predicted_fail = {"a", "b", "e", "f"}  # written before any check ran

reports = {key: check_submission(path) for key, path in candidate_paths.items()}
actual_fail = {key for key, report in reports.items() if not report.ok}
for key, report in reports.items():
    print(key, "ok" if report.ok else report.findings)

fixes = {
    "a": "def mean_speed(speeds: List[float]) -> float: return sum(speeds) / len(speeds)",
    "b": "def short_name(name: str) -> str: return name[len('agent-'):] if name.startswith('agent-') else name",
    "e": "def merged(a: dict, b: dict) -> dict: return {**a, **b}",
    "f": "def describe(speed: float) -> str: return 'stopped' if speed == 0.0 else 'moving'",
}

assert actual_fail == predicted_fail
assert {key for key in CANDIDATES if key not in actual_fail} == {"c", "d", "g"}
assert sorted(fixes) == sorted(actual_fail)
for key, fix in fixes.items():
    fixed_path = workdir / f"fixed_{key}.py"
    with open(fixed_path, "w") as handle:
        handle.write(starter_source + "\n\n" + fix + "\n")
    fixed_report = check_submission(fixed_path, laps=1)
    assert fixed_report.ok, (key, fixed_report.findings)

## Exercise 8 - Challenge: predict, then measure, a late crash

Under the pinned rule the crash throws away everything driven so far and adds the 1.0 s hold, so the total is the clean race, plus the time of the crash, plus 1.0 s.
The lesson's crash came 0.60 s after the sabotage began; on the oval at a steady speed the car is in the same state at 120 s as at 60 s, so expect a crash at 120.60 s: 223.50 + 120.60 + 1.0 = 345.10 s.
"A crash costs a lap" would predict 223.50 + 223.50 / 3 = 298.00 s.

The two rules cost the same when `t_crash + 1.0 = 223.50 / 3 = 74.50`, at 73.50 s.
Lap 1 on the oval takes about 76.9 s, so **every** crash in lap 2 or lap 3 costs more under the pinned rule than a lap.

In [ ]:
SABOTAGE = """

CRASH_AT = 60.0  # seconds: when the sabotage starts


_Starter = RoarCompetitionSolution


class RoarCompetitionSolution(_Starter):
    \"\"\"The starter, sabotaged: from CRASH_AT it steers right at full throttle until it crashes.\"\"\"

    async def initialize(self) -> None:
        await super().initialize()
        self.steps = 0
        self.crashed = False

    async def step(self) -> None:
        impulse = self.collision_sensor.get_last_gym_observation()
        if float(np.linalg.norm(impulse)) > 100.0:  # the first step after a respawn
            self.crashed = True
        await super().step()
        if not self.crashed and self.steps * 0.05 >= CRASH_AT:
            await self.vehicle.apply_action({
                "throttle": 1.0, "steer": 0.25, "brake": 0.0,
                "hand_brake": 0.0, "reverse": 0, "target_gear": 0,
            })
        self.steps += 1
"""

saboteur_path = workdir / "saboteur.py"
with open(STARTER) as source, open(saboteur_path, "w") as target:
    target.write(source.read() + SABOTAGE)

clean_oval = 223.50
predicted_total = clean_oval + (120.0 + 0.60) + 1.0
costs_a_lap_prediction = clean_oval + clean_oval / 3
late_path = workdir / "saboteur_120.py"
write_variant(saboteur_path, late_path, {"CRASH_AT": "120.0"})
late_crash = run_race(late_path, "oval", car=KinematicCar(jitter=0.0))
t_crash = late_crash.respawn_times[0]
held_rows = int(np.sum(late_crash.telemetry["held"] == 1))
prediction_error = late_crash.total_time - predicted_total
breakeven_crash_time = clean_oval / 3 - 1.0
verdict = (
    "The pinned rule punishes a lap-2 crash more, because any crash after 73.5 s costs more than a lap "
    "and a crash late in lap 3 costs nearly a whole extra race, so risk is worth less and less as the "
    "race goes on: late in a race, drive safe."
)

print(late_crash, "| crash at", round(t_crash, 2), "s |", held_rows, "held rows")
print(f"predicted {predicted_total:.2f} s, measured {late_crash.total_time:.2f} s, error {prediction_error:.2f} s")
print(f"'costs a lap' predicted {costs_a_lap_prediction:.2f} s; break-even crash time {breakeven_crash_time:.2f} s")
assert late_crash.finished and late_crash.collisions == 1 and len(late_crash.respawn_times) == 1
assert abs(t_crash - 120.60) < 1e-9
assert held_rows == 20  # 1.0 s of respawn hold at 0.05 s per tick
assert abs(prediction_error) < CONTROL_DT
assert abs(costs_a_lap_prediction - 298.0) < 1e-9 and abs(breakeven_crash_time - 73.5) < 1e-9
assert late_crash.total_time > costs_a_lap_prediction
# A lap-2 crash happens after lap 1 ends, which is already past the break-even time.
assert sum(run_race(STARTER, "oval", car=KinematicCar(jitter=0.0)).sector_splits[0]) > breakeven_crash_time

## Exercise 9 - Challenge: a faster variant, accepted honestly

A higher cap is a real gain on the oval: about 18 s, more than 50 times the 0.35 s spread, with every run clean.
It also passes `check_submission` and finishes `monza-like` clean, still more than 4.8 m inside the narrowest half-width.
On these tracks 22 m/s is far below any corner limit, so the extra speed costs no safety yet; this is not a licence to raise it blindly.

In [ ]:
variant_path = workdir / "target22.py"
write_variant(STARTER, variant_path, {"TARGET_SPEED": "22.0"})
jittered = lambda: KinematicCar(jitter=0.05)
starter_runs = repeat_races(STARTER, "oval", n=5, car_factory=jittered)
variant_runs = repeat_races(variant_path, "oval", n=5, car_factory=jittered)
variant_better = noise_aware_better(variant_runs.times, starter_runs.times)
median_gain = starter_runs.median - variant_runs.median
variant_report = check_submission(variant_path)
variant_clean = variant_runs.all_finished and variant_runs.collisions == 0 and variant_report.ok
monza_variant = run_race(variant_path, "monza-like", car=KinematicCar(jitter=0.05), seed=0)
monza_margin = float(np.min(load_track("monza-like").half_width) - np.max(np.abs(monza_variant.telemetry["offset"])))
accept = variant_better and variant_clean and monza_variant.finished and monza_variant.collisions == 0 and monza_margin > 0
reason = (
    f"Accept: the median drops by {median_gain:.2f} s, far more than the {max(starter_runs.spread, variant_runs.spread):.2f} s "
    f"spread, all 5 oval runs and the monza-like run are clean, the checker passes, and the car stays "
    f"{monza_margin:.2f} m inside the narrowest half-width."
)

print("starter:", starter_runs, "| variant:", variant_runs)
print("variant better?", variant_better, "| median gain:", round(median_gain, 2), "s | clean?", variant_clean)
print(monza_variant, "| margin:", round(monza_margin, 2), "m")
print(reason)
# Predict the gain: both run at their settling speeds, which scale with the target.
qa, qb = 4e-4, 6.0 * 0.05
settle = {target: float((-qb + np.sqrt(qb * qb + 4 * qa * qb * target)) / (2 * qa)) for target in (20.0, 22.0)}
predicted_gain = 3 * load_track("oval").length * (1 / settle[20.0] - 1 / settle[22.0])
print("predicted gain from settling speeds:", round(predicted_gain, 2), "s")
assert abs(starter_runs.median - 220.75) < 1e-9  # the lesson's seeds 0-4
assert variant_better and not noise_aware_better(starter_runs.times, variant_runs.times)
assert abs(median_gain - predicted_gain) < 2.0
assert variant_clean and monza_variant.finished and monza_variant.collisions == 0
assert monza_variant.total_time < 870.30  # faster than the starter's lesson race too
assert 0.0 < monza_margin < float(np.min(load_track("monza-like").half_width))
assert accept